# Species Showcase: Before/After Algorithm

Find the **best-performing threatened species** and visualize:
- **Blue:** Observed occurrences (GBIF ground truth)
- **Green:** Correct predictions (TP – predicted & observed)
- **Red:** Wrong predictions (FP – predicted but not observed)
- **Orange:** Missed (FN – observed but not predicted)

**Requires:** Run `species_predictor.ipynb` first (model + scaler in output/). Loads data from S3.

In [ ]:
%pip install -q pandas numpy scikit-learn torch h3 pyarrow boto3 geopandas folium xgboost

In [27]:
# Same config as species_predictor.ipynb
COUNTRY = "ES"
H3_RES = 7
PARENT_RES = 5
YEAR_START, YEAR_END = 2015, 2024
S3_BUCKET = "ie-datalake"
GOLD_H3_MAPPING = "gold/gbif_species_h3_mapping"
GOLD_SPECIES_DIM = "gold/gbif_species_dim"
GOLD_OSM_HEX = "gold/osm_hex_features"
GOLD_CELL_METRICS = "gold/gbif_cell_metrics"
GOLD_NATURE2000 = "gold/nature2000_cell_protection"
GOLD_GEE_TERRAIN = "gold/gee_hex_terrain"
GEE_TERRAIN_SNAPSHOT = "2019"
NATURE2000_SNAPSHOT_DATE = "2026-02-27"
AWS_PROFILE = "486717354268_PowerUserAccess"
OUTPUT_DIR = "output"
MODEL_PATH = f"{OUTPUT_DIR}/species_predictor.pt"
XGB_MODEL_PATH = f"{OUTPUT_DIR}/species_predictor_xgb.pkl"
TARGET_SPECIES_PATH = f"{OUTPUT_DIR}/target_species.csv"
SCALER_PATH = f"{OUTPUT_DIR}/scaler.pkl"
USE_XGBOOST = True
TARGET_YEAR = 2024
FEATURE_YEARS = (2019, 2023)
PRESENCE_THRESHOLD = 2
N_THREATENED = 20
HIDDEN_SIZES = [256, 128]
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
PRED_THRESHOLD = 0.5  # threshold for positive prediction
MIN_OCCURRENCES = 50  # species must have at least this many observed hexes to be considered

## 1) S3 connection + load data

In [28]:
import pickle
import logging
from pathlib import Path

import boto3
import h3
import numpy as np
import pandas as pd
import pyarrow.fs as pafs
import torch
import torch.nn as nn

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("04")

session = boto3.Session(profile_name=AWS_PROFILE)
creds = session.get_credentials().get_frozen_credentials()
fs_read = pafs.S3FileSystem(access_key=creds.access_key, secret_key=creds.secret_key, session_token=creds.token, region=session.region_name or "eu-west-2")

from data_loader import load_all

# Use target_species_ids from model's target_species.csv (must match model training order)
target_path = Path(TARGET_SPECIES_PATH)
if not target_path.exists():
    target_path = Path("gbif_species_predictor") / TARGET_SPECIES_PATH
df_target = pd.read_csv(target_path)
model_target_species_ids = df_target["species_id"].astype(int).tolist()

config = {
    "COUNTRY": COUNTRY, "H3_RES": H3_RES, "PARENT_RES": PARENT_RES, "S3_BUCKET": S3_BUCKET,
    "GOLD_H3_MAPPING": GOLD_H3_MAPPING, "GOLD_OSM_HEX": GOLD_OSM_HEX, "GOLD_CELL_METRICS": GOLD_CELL_METRICS,
    "GOLD_NATURE2000": GOLD_NATURE2000, "GOLD_GEE_TERRAIN": GOLD_GEE_TERRAIN, "GOLD_SPECIES_DIM": GOLD_SPECIES_DIM,
    "GEE_TERRAIN_SNAPSHOT": GEE_TERRAIN_SNAPSHOT, "NATURE2000_SNAPSHOT_DATE": NATURE2000_SNAPSHOT_DATE,
    "TARGET_YEAR": TARGET_YEAR, "FEATURE_YEARS": FEATURE_YEARS, "PRESENCE_THRESHOLD": PRESENCE_THRESHOLD,
    "N_THREATENED": N_THREATENED, "TRAIN_FRAC": TRAIN_FRAC, "VAL_FRAC": VAL_FRAC,
    "target_species_ids": model_target_species_ids,
}
data = load_all(fs_read, config)
hex_test = data["hex_test"]
Y_test = data["Y_test"]
target_species_ids = data["target_species_ids"]
species_names = data["species_names"]
FEATURE_COLS = data["FEATURE_COLS"]
h3_col = data["h3_col"]
df_feat = data["df_feat"]

scaler_path = Path(SCALER_PATH)
if not scaler_path.exists():
    scaler_path = Path("gbif_species_predictor") / SCALER_PATH
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler not found. Run species_predictor.ipynb first. Tried: {SCALER_PATH}")
with open(scaler_path, "rb") as f:
    scaler_obj = pickle.load(f)
scaler = scaler_obj["scaler"]
feat_cols = scaler_obj.get("feature_cols", FEATURE_COLS)
for c in feat_cols:
    if c not in df_feat.columns:
        df_feat[c] = 0.0
X_test = scaler.transform(df_feat.loc[data["test_mask"], feat_cols].fillna(0).replace([np.inf, -np.inf], 0).values.astype(np.float32))
print(f"Loaded: {len(hex_test)} test hexes, {len(target_species_ids)} species")

Found credentials in shared credentials file: ~/.aws/credentials
Loading data...


Loaded: 2968 test hexes, 20 species


## 2) Load model, run inference on test set

In [29]:
xgb_path = Path(XGB_MODEL_PATH)
if not xgb_path.exists():
    xgb_path = Path("gbif_species_predictor") / XGB_MODEL_PATH
if USE_XGBOOST and xgb_path.exists():
    with open(xgb_path, "rb") as f:
        xgb_obj = pickle.load(f)
    xgb_multi = xgb_obj["model"]
    probs_test = np.column_stack([p[:, 1] for p in xgb_multi.predict_proba(X_test)]).astype(np.float32)
    print("Loaded XGBoost from", xgb_path)
elif USE_XGBOOST:
    print("XGB_MODEL_PATH not found, falling back to MLP")
    USE_XGBOOST = False
if not USE_XGBOOST:
    model_path = Path(MODEL_PATH)
    if not model_path.exists():
        model_path = Path("gbif_species_predictor") / MODEL_PATH
    class MultiLabelMLP(nn.Module):
        def __init__(self, n_in, n_out, hidden_sizes=(256, 128), dropout=0.3):
            super().__init__()
            layers = []
            prev = n_in
            for h in hidden_sizes:
                layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
                prev = h
            self.backbone = nn.Sequential(*layers)
            self.head = nn.Linear(prev, n_out)
        def forward(self, x): return self.head(self.backbone(x))
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    model = MultiLabelMLP(X_test.shape[1], len(target_species_ids), HIDDEN_SIZES, 0.0)
    model.load_state_dict(ckpt["model"])
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test, dtype=torch.float32))
        probs_test = torch.sigmoid(logits).numpy()
    print("Loaded MLP from", model_path)

pred_binary = (probs_test >= PRED_THRESHOLD).astype(np.float32)
print("Predictions shape:", probs_test.shape)

Loaded XGBoost from output/species_predictor_xgb.pkl
Predictions shape: (2968, 20)


## 3) Per-species metrics, pick best

In [30]:
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

species_metrics = []
for j, sid in enumerate(target_species_ids):
    y_true = Y_test[:, j]
    y_pred = probs_test[:, j]
    y_bin = pred_binary[:, j]
    n_pos = int(y_true.sum())
    if n_pos < MIN_OCCURRENCES:
        continue
    pr_auc = average_precision_score(y_true, y_pred)
    prec = precision_score(y_true, y_bin, zero_division=0)
    rec = recall_score(y_true, y_bin, zero_division=0)
    f1 = f1_score(y_true, y_bin, zero_division=0)
    species_metrics.append({
        "j": j,
        "species_id": sid,
        "species_name": species_names.get(sid, str(sid)),
        "n_observed": n_pos,
        "pr_auc": pr_auc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
    })

df_metrics = pd.DataFrame(species_metrics)
if df_metrics.empty:
    raise ValueError(f"No species with >= {MIN_OCCURRENCES} occurrences. Lower MIN_OCCURRENCES or check data.")
best_idx = df_metrics["pr_auc"].idxmax()
best_row = df_metrics.loc[best_idx]
best_species_j = int(best_row["j"])
best_species_id = int(best_row["species_id"])
best_species_name = str(best_row["species_name"])

print(f"Best species: {best_species_name} (PR-AUC={best_row['pr_auc']:.3f}, Precision={best_row['precision']:.2%}, Recall={best_row['recall']:.2%})")
print(df_metrics.sort_values("pr_auc", ascending=False).head(5).to_string())

Best species: Ichthyaetus audouinii (PR-AUC=0.770, Precision=70.27%, Recall=63.80%)
   j species_id           species_name  n_observed    pr_auc  precision    recall        f1
3  3    7406504  Ichthyaetus audouinii         163  0.770405   0.702703  0.638037  0.668810
5  5    2480327   Pluvialis squatarola         105  0.674678   0.739726  0.514286  0.606742
6  6    2474921             Otis tarda         110  0.651252   0.866667  0.236364  0.371429
7  7    2480510       Aquila adalberti         156  0.596173   0.818182  0.346154  0.486486
1  1    2498255          Aythya ferina         107  0.565142   0.571429  0.448598  0.502618


## 4) Build hex sets: observed, TP, FP, FN

In [31]:
j = best_species_j
observed = set(hex_test[Y_test[:, j] > 0])  # ground truth positive
predicted = set(hex_test[pred_binary[:, j] > 0])  # model said positive

tp_hexes = observed & predicted
fp_hexes = predicted - observed
fn_hexes = observed - predicted

n_tp, n_fp, n_fn = len(tp_hexes), len(fp_hexes), len(fn_hexes)
n_obs = len(observed)
accuracy_pct = 100 * n_tp / n_obs if n_obs else 0  # % of observed we correctly predicted
precision_pct = 100 * n_tp / (n_tp + n_fp) if (n_tp + n_fp) else 0
recall_pct = 100 * n_tp / (n_tp + n_fn) if (n_tp + n_fn) else 0

print(f"{best_species_name}")
print(f"  Observed (blue): {n_obs} hexes")
print(f"  Correct (green): {n_tp} TP")
print(f"  Wrong (red): {n_fp} FP (predicted, not observed)")
print(f"  Missed (orange): {n_fn} FN (observed, not predicted)")
print(f"  Accuracy (TP/observed): {accuracy_pct:.1f}%")
print(f"  Precision: {precision_pct:.1f}%")
print(f"  Recall: {recall_pct:.1f}%")

Ichthyaetus audouinii
  Observed (blue): 163 hexes
  Correct (green): 104 TP
  Wrong (red): 44 FP (predicted, not observed)
  Missed (orange): 59 FN (observed, not predicted)
  Accuracy (TP/observed): 63.8%
  Precision: 70.3%
  Recall: 63.8%


## 5) Folium map: hexagons on Spain (TP / FP / FN)

In [32]:
import folium

def h3_to_folium_coords(h):
    """H3 hex to [[lat, lng], ...] for Folium (closed polygon)."""
    b = h3.cell_to_boundary(h)
    coords = [[p[0], p[1]] for p in b]  # (lat, lng) -> [lat, lng]
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    return coords

# Spain center
m = folium.Map(location=[40.4, -3.7], zoom_start=6, tiles="CartoDB positron")

# Layer groups for toggle
fg_tp = folium.FeatureGroup(name=f"Correct (TP) – {n_tp}", show=True)
fg_fp = folium.FeatureGroup(name=f"Wrong (FP) – {n_fp}", show=True)
fg_fn = folium.FeatureGroup(name=f"Missed (FN) – {n_fn}", show=True)

for h in tp_hexes:
    try:
        folium.Polygon(locations=h3_to_folium_coords(h), color="#22c55e", weight=1, fill=True, fill_color="#22c55e", fill_opacity=0.7, popup="TP").add_to(fg_tp)
    except Exception:
        pass
for h in fp_hexes:
    try:
        folium.Polygon(locations=h3_to_folium_coords(h), color="#ef4444", weight=1, fill=True, fill_color="#ef4444", fill_opacity=0.7, popup="FP").add_to(fg_fp)
    except Exception:
        pass
for h in fn_hexes:
    try:
        folium.Polygon(locations=h3_to_folium_coords(h), color="#f97316", weight=1, fill=True, fill_color="#f97316", fill_opacity=0.7, popup="FN").add_to(fg_fn)
    except Exception:
        pass

fg_tp.add_to(m)
fg_fp.add_to(m)
fg_fn.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_html = f"""
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background: white; padding: 10px; border-radius: 5px; box-shadow: 0 2px 6px rgba(0,0,0,0.3);">
  <p><b>{best_species_name}</b></p>
  <p>Accuracy {accuracy_pct:.1f}% | Precision {precision_pct:.1f}% | Recall {recall_pct:.1f}%</p>
  <p style="color:#22c55e">● Correct (TP): {n_tp}</p>
  <p style="color:#ef4444">● Wrong (FP): {n_fp}</p>
  <p style="color:#f97316">● Missed (FN): {n_fn}</p>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
out_path = f"{OUTPUT_DIR}/species_showcase_{best_species_id}.html"
m.save(out_path)
print(f"Saved: {out_path}")
m

Saved: output/species_showcase_7406504.html


## 6) Display map (if not shown above)

In [ ]:
# Open saved HTML in browser if needed
import webbrowser
html_path = Path(OUTPUT_DIR) / f"species_showcase_{best_species_id}.html"
if html_path.exists():
    webbrowser.open(f"file://{html_path.absolute()}")